# Practical 11: Convolutional Autoencoder on MNIST Dataset

**Problem Statement:** Implement a Convolutional Autoencoder to learn compressed image representations and reconstruct handwritten digits.

**Activities:**
1. Build encoder-decoder architecture
2. Train autoencoder
3. Compare original and reconstructed images

**Dataset:** MNIST — 70,000 grayscale images of handwritten digits (0-9), 28x28 pixels.

## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 2. Build Encoder-Decoder Architecture

The encoder compresses each 28x28 image down to a small spatial representation using convolution and max-pooling. The decoder mirrors this with convolution and upsampling layers to reconstruct the original image size.

In [ ]:
input_img = keras.layers.Input(shape=(28, 28, 1))

# Encoder
x = keras.layers.Conv2D(32, 3, activation='relu', padding='same')(input_img)
x = keras.layers.MaxPooling2D(2, padding='same')(x)
x = keras.layers.Conv2D(16, 3, activation='relu', padding='same')(x)
encoded = keras.layers.MaxPooling2D(2, padding='same')(x)

# Decoder
x = keras.layers.Conv2D(16, 3, activation='relu', padding='same')(encoded)
x = keras.layers.UpSampling2D(2)(x)
x = keras.layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = keras.layers.UpSampling2D(2)(x)
decoded = keras.layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)

autoencoder = keras.Model(input_img, decoded)
autoencoder.summary()

## 3. Train Autoencoder

The autoencoder is trained to reconstruct its own input, so no labels are used.

In [ ]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_test, X_test),
    epochs=15,
    batch_size=128,
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Reconstruction Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 4. Compare Original and Reconstructed Images

In [ ]:
reconstructed = autoencoder.predict(X_test[:10], verbose=0)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(X_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')

    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_title('Original', loc='left', fontsize=10)
axes[1, 0].set_title('Reconstructed', loc='left', fontsize=10)
plt.show()

## Conclusion

In this practical, we:
- Built a convolutional encoder-decoder architecture
- Trained the autoencoder to reconstruct MNIST digits from a compressed representation
- Compared original and reconstructed images to assess reconstruction quality